In [7]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:


IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 1  #For Grayscale(3 for RGB)


In [9]:

train_path = "dataset2/train"
train_label_path = "dataset2/train_label"
valid_path = "dataset2/val"
valid_label_path = "dataset2/val_label"



In [ ]:


def load_data(image_dir, label_dir, label_extension=".png"):
    images = []
    labels = []
    for image_name in os.listdir(image_dir):
        
        image_path = os.path.join(image_dir, image_name)
        label_name = os.path.splitext(image_name)[0] + label_extension 
        label_path = os.path.join(label_dir, label_name)

       
        if not os.path.exists(label_path):
            print(f"Label file missing for {image_name}, skipping...")
            continue

        image = load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode='grayscale')
        image = img_to_array(image) / 255.0
        images.append(image)

        label = load_img(label_path, target_size=(IMG_HEIGHT, IMG_WIDTH), color_mode='grayscale')
        label = img_to_array(label) / 255.0
        labels.append(label)

    return np.array(images), np.array(labels)


In [10]:


def load_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error loading image: {image_path}")
    return image


def resize_images(images, size=(320, 320)):
    resized_images = [cv2.resize(image, size) for image in images]
    return np.array(resized_images)


def load_data(data_path, label_path):
    data_files = sorted(os.listdir(data_path))
    label_files = sorted(os.listdir(label_path))

    images = []
    labels = []

    for data_file, label_file in zip(data_files, label_files):
       
        image_path = os.path.join(data_path, data_file)
        label_image_path = os.path.join(label_path, label_file)

        
        print(f"Loading image: {image_path}")
        print(f"Loading label: {label_image_path}")

        
        image = load_image(image_path)
        label = load_image(label_image_path)

        if image is not None and label is not None:
            
            image = cv2.resize(image, (320, 320))  # Resize to 320x320
            label = cv2.resize(label, (320, 320))  

            
            label = cv2.cvtColor(label, cv2.COLOR_BGR2GRAY)
            label = np.expand_dims(label, axis=-1)  

            images.append(image)
            labels.append(label)
        else:
            print(f"Skipping {data_file} and {label_file} due to load error.")

    images = np.array(images)
    labels = np.array(labels)

    return images, labels


X_train, Y_train = load_data(train_path, train_label_path)
X_valid, Y_valid = load_data(valid_path, valid_label_path)
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_valid:", X_valid.shape)
print("Y_valid:", Y_valid.shape)


if X_train.shape[0] == 0 or X_valid.shape[0] == 0:
    raise ValueError("No valid images were loaded. Check the file paths or images.")

X_train = X_train.astype("float32") / 255.0 #Normalization
X_valid = X_valid.astype("float32") / 255.0
Y_train = Y_train.astype("float32") / 255.0
Y_valid = Y_valid.astype("float32") / 255.0


Y_train = np.expand_dims(Y_train, axis=-1)
Y_valid = np.expand_dims(Y_valid, axis=-1)



Loading image: dataset2/train\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1828.png
Loading label: dataset2/train_label\Town04_Clear_Noon_09_09_2020_14_57_22_frame_0_label.png
Loading image: dataset2/train\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1829.png
Loading label: dataset2/train_label\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1000_label.png
Loading image: dataset2/train\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1830.png
Loading label: dataset2/train_label\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1001_label.png
Loading image: dataset2/train\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1831.png
Loading label: dataset2/train_label\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1002_label.png
Loading image: dataset2/train\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1832.png
Loading label: dataset2/train_label\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1003_label.png
Loading image: dataset2/train\Town04_Clear_Noon_09_09_2020_14_57_22_frame_1833.png
Loading label: dataset2/train_

In [ ]:
def build_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(320, 320, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        UpSampling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        UpSampling2D((2, 2)),
        Conv2D(1, (3, 3), activation='sigmoid', padding='same')  
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow(X_train, Y_train, batch_size=8)
valid_generator = valid_datagen.flow(X_valid, Y_valid, batch_size=8)


model = build_model()
model.summary()


early_stopping = EarlyStopping(monitor='val_loss', patience=5)

history = model.fit(train_generator, validation_data=valid_generator, epochs=10, callbacks=[early_stopping])

#Saving here
model.save("lane_detection_model.h5")

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 320, 320, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 160, 160, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 160, 160, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 80, 80, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 80, 80, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 160, 160, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 160, 160, 64)   │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 320, 320, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 320, 320, 1)    │           577 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 167,617 (654.75 KB)

 Trainable params: 167,617 (654.75 KB)

 Non-trainable params: 0 (0.00 B)

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 134s 788ms/step - accuracy: 0.9520 - loss: 0.1393 - val_accuracy: 0.9854 - val_loss: 8.5344e-04
Epoch 2/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 150s 899ms/step - accuracy: 0.9854 - loss: 8.6303e-04 - val_accuracy: 0.9854 - val_loss: 8.4684e-04
Epoch 3/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 135s 808ms/step - accuracy: 0.9855 - loss: 8.5710e-04 - val_accuracy: 0.9854 - val_loss: 8.4129e-04
Epoch 4/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 254s 2s/step - accuracy: 0.9855 - loss: 8.5143e-04 - val_accuracy: 0.9854 - val_loss: 8.3585e-04
Epoch 5/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 156s 935ms/step - accuracy: 0.9854 - loss: 8.5438e-04 - val_accuracy: 0.9854 - val_loss: 8.2878e-04
Epoch 6/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 133s 797ms/step - accuracy: 0.9855 - loss: 8.4262e-04 - val_accuracy: 0.9854 - val_loss: 8.2741e-04
Epoch 7/10
167/167 ━━━━━━━━━━━━━━━━━━━━ 467s 3s/step - accuracy: 0.9854 - loss: 8.4827e-04 - val_accuracy: 0.9854 - val_loss: 8.1844e-04
Epoch 8/10
167/167 ━━━━━━━━━━━

In [ ]:


val_loss, val_accuracy = model.evaluate(valid_generator)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")

output_dir = "predictions"
os.makedirs(output_dir, exist_ok=True)


predictions = model.predict(X_valid)

for i in range(10):  
    
    original_image = X_valid[i]

    
    true_label = Y_valid[i].squeeze()  

    predicted_label = predictions[i].squeeze()  

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
plt.title("Original Image")
plt.imshow(original_image)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Actual Lane Mask")
plt.imshow(true_label, cmap="gray")
plt.axis("off")
plt.title("predicted lanes")
plt.imshow(predicted_label, cmap="gray")
plt.axis("off")

 

    output_path = os.path.join(output_dir, f"prediction_{i}.png")
    plt.savefig(output_path)
    plt.close()

    print(f"Saved prediction visualization: {output_path}")

model.save("lane_detection_model.h5")
print("Model saved as lane_detection_model.h5")


NameError: name 'model' is not defined